# BirdCLEF+ 2026 - EfficientNet-B0 Submission

Purpose: load the trained EfficientNet-B0 artifact, run soundscape inference, and write `submission.csv` for Kaggle.

Artifacts are written to `/kaggle/working/artifacts/submission`.


## 1. Setup


In [1]:
from pathlib import Path
import json
import os
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)


class CFG:
    seed = 42
    competition_name = "birdclef-2026"
    data_root = None
    artifact_dir = Path("/kaggle/working/artifacts")


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


def find_data_root() -> Path:
    candidates = [
        Path("/kaggle/input/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack"),
        Path("data/raw/birdclef-2026"),
    ]
    for path in candidates:
        if (path / "train.csv").exists():
            return path
    input_root = Path("/kaggle/input")
    if input_root.exists():
        matches = list(input_root.glob("**/train.csv"))
        if matches:
            return matches[0].parent
    raise FileNotFoundError("Could not find train.csv. Attach the BirdCLEF 2026 dataset.")


def read_optional_csv(path: Path) -> pd.DataFrame | None:
    return pd.read_csv(path) if path.exists() else None


seed_everything(CFG.seed)
CFG.data_root = find_data_root()
CFG.artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Data root: {CFG.data_root}")
print(f"Artifacts: {CFG.artifact_dir}")

try:
    import timm
except ImportError:
    import sys
    !{sys.executable} -m pip install -q timm
    import timm

import librosa
import zipfile
import torch
from torch import nn
from tqdm.auto import tqdm
from IPython.display import FileLink

torch.set_num_threads(min(2, os.cpu_count() or 1))


class CFG(CFG):
    artifact_dir = Path("/kaggle/working/artifacts/submission")
    sample_rate = 32000
    duration = 5.0
    n_fft = 2048
    hop_length = 512
    n_mels = 128
    fmin = 20
    fmax = 16000
    backbone = "efficientnet_b0"
    batch_size = 32
    checkpoint_path = None
    labels_path = None


CFG.artifact_dir.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


Data root: /kaggle/input/competitions/birdclef-2026
Artifacts: /kaggle/working/artifacts
Device: cpu


## 2. Locate Competition Files And Model Artifact


In [2]:
def find_file(filename: str, roots: list[Path]) -> Path:
    for root in roots:
        if not root.exists():
            continue
        direct = root / filename
        if direct.exists():
            return direct
        matches = list(root.glob(f"**/{filename}"))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"Could not find {filename}. Attach the EfficientNet artifact dataset.")


artifact_roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working/artifacts/effnet_b0"),
    Path("artifacts/effnet_b0"),
]
sample_path = CFG.data_root / "sample_submission.csv"
checkpoint_path = Path(CFG.checkpoint_path) if CFG.checkpoint_path else find_file("best_effnet_b0.pt", artifact_roots)
labels_path = Path(CFG.labels_path) if CFG.labels_path else find_file("labels.json", artifact_roots)

sample_submission = pd.read_csv(sample_path)
idx_to_label = {int(k): v for k, v in json.loads(labels_path.read_text()).items()}
labels = [idx_to_label[i] for i in sorted(idx_to_label)]
label_to_idx = {label: i for i, label in enumerate(labels)}
target_columns = [c for c in sample_submission.columns if c != "row_id"]

print(f"Sample submission: {sample_path}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Labels: {labels_path}")
print(f"Rows: {len(sample_submission):,}")
print(f"Submission target columns: {len(target_columns):,}")
print(f"Model classes: {len(labels):,}")
display(sample_submission.head())


Sample submission: /kaggle/input/competitions/birdclef-2026/sample_submission.csv
Checkpoint: /kaggle/input/models/tuannm3823/birdclef-2026-effnet-b0/pytorch/baseline-v1/1/best_effnet_b0.pt
Labels: /kaggle/input/models/tuannm3823/birdclef-2026-effnet-b0/pytorch/baseline-v1/1/labels.json
Rows: 3
Submission target columns: 234
Model classes: 206


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,22967,22973,22983,22985,23150,23154,23158,23176,23724,24279,24285,24287,24321,244024,25073,25092,25214,326272,41970,43435,47144,47158son01,47158son02,47158son03,47158son04,47158son05,47158son06,47158son07,47158son08,47158son09,...,sobtyr1,socfly1,sofspi1,souant1,soulap1,souscr1,spbant3,spispi1,sptnig1,squcuc1,stbwoo2,strcuc1,strher2,strowl1,swthum1,swtman1,tattin1,thlwre1,toctou1,trokin,trsowl,undtin1,varant1,watjac1,wesfie1,wfwduc1,whbant2,whbwar2,whiwoo1,whlspi1,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274


## 3. Build Model And Audio Helpers


In [3]:
class BirdClassifier(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.model = timm.create_model(
            CFG.backbone,
            pretrained=False,
            in_chans=1,
            num_classes=num_classes,
        )

    def forward(self, x):
        return self.model(x)


def torch_load(path: Path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


checkpoint = torch_load(checkpoint_path)
model = BirdClassifier(num_classes=len(labels)).to(device)
model.load_state_dict(checkpoint["model"])
model.eval()


def row_to_stem_and_end_time(row_id: str) -> tuple[str, float]:
    stem, sep, end = str(row_id).rpartition("_")
    if sep and end.replace(".", "", 1).isdigit():
        return stem, float(end)
    return str(row_id), CFG.duration


def build_audio_index() -> dict[str, Path]:
    candidates = [
        CFG.data_root / "test_soundscapes",
        CFG.data_root / "test_audio",
        CFG.data_root / "train_soundscapes",
    ]
    audio_index = {}
    for folder in candidates:
        if folder.exists():
            for ext in ("*.ogg", "*.wav", "*.flac", "*.mp3"):
                for path in folder.glob(ext):
                    audio_index[path.stem] = path
    return audio_index


audio_index = build_audio_index()
print(f"Indexed audio files: {len(audio_index):,}")


Indexed audio files: 10,658


## 4. Run Inference


In [4]:
def load_audio_segment(path: Path, end_time: float) -> np.ndarray:
    offset = max(0.0, float(end_time) - CFG.duration)
    target_len = int(CFG.sample_rate * CFG.duration)
    y, _ = librosa.load(path, sr=CFG.sample_rate, mono=True, offset=offset, duration=CFG.duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y[:target_len].astype(np.float32)


def audio_to_mel(y: np.ndarray) -> np.ndarray:
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=CFG.sample_rate,
        n_fft=CFG.n_fft,
        hop_length=CFG.hop_length,
        n_mels=CFG.n_mels,
        fmin=CFG.fmin,
        fmax=CFG.fmax,
        power=2.0,
    )
    mel = librosa.power_to_db(mel, ref=np.max)
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)
    return mel.astype(np.float32)


def predict_batch(batch: list[np.ndarray]) -> np.ndarray:
    x = torch.from_numpy(np.stack(batch)).unsqueeze(1).to(device)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    return probs


submission = sample_submission.copy()
for col in target_columns:
    submission[col] = 0.0

batch = []
batch_rows = []
missing_audio = []

for row_idx, row_id in tqdm(list(enumerate(submission["row_id"])), desc="inference"):
    stem, end_time = row_to_stem_and_end_time(row_id)
    audio_path = audio_index.get(stem)
    if audio_path is None:
        missing_audio.append(row_id)
        continue
    batch.append(audio_to_mel(load_audio_segment(audio_path, end_time)))
    batch_rows.append(row_idx)
    if len(batch) == CFG.batch_size:
        probs = predict_batch(batch)
        for local_idx, submit_idx in enumerate(batch_rows):
            for label, class_idx in label_to_idx.items():
                if label in target_columns:
                    submission.loc[submit_idx, label] = probs[local_idx, class_idx]
        batch = []
        batch_rows = []

if batch:
    probs = predict_batch(batch)
    for local_idx, submit_idx in enumerate(batch_rows):
        for label, class_idx in label_to_idx.items():
            if label in target_columns:
                submission.loc[submit_idx, label] = probs[local_idx, class_idx]

missing_audio_path = CFG.artifact_dir / "missing_test_audio.json"
missing_audio_path.write_text(json.dumps(missing_audio, indent=2), encoding="utf-8")
print(f"Missing audio rows: {len(missing_audio):,}")
display(submission.head())


inference:   0%|          | 0/3 [00:00<?, ?it/s]

Missing audio rows: 3


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,22967,22973,22983,22985,23150,23154,23158,23176,23724,24279,24285,24287,24321,244024,25073,25092,25214,326272,41970,43435,47144,47158son01,47158son02,47158son03,47158son04,47158son05,47158son06,47158son07,47158son08,47158son09,...,sobtyr1,socfly1,sofspi1,souant1,soulap1,souscr1,spbant3,spispi1,sptnig1,squcuc1,stbwoo2,strcuc1,strher2,strowl1,swthum1,swtman1,tattin1,thlwre1,toctou1,trokin,trsowl,undtin1,varant1,watjac1,wesfie1,wfwduc1,whbant2,whbwar2,whiwoo1,whlspi1,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,BC2026_Test_0001_S05_20250227_010002_10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,BC2026_Test_0001_S05_20250227_010002_15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 5. Save Submission And Package Artifacts


In [5]:
submission_path = Path("/kaggle/working/submission.csv")
submission.to_csv(submission_path, index=False)
submission.to_csv(CFG.artifact_dir / "submission.csv", index=False)

zip_path = Path("/kaggle/working/birdclef_effnet_b0_submission_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(submission_path, arcname="submission.csv")
    for path in sorted(CFG.artifact_dir.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(CFG.artifact_dir.parent))

print(f"Wrote submission: {submission_path}")
print(f"Wrote artifact zip: {zip_path} ({zip_path.stat().st_size / 1024 / 1024:.2f} MB)")
display(FileLink(submission_path))
display(FileLink(zip_path))


Wrote submission: /kaggle/working/submission.csv
Wrote artifact zip: /kaggle/working/birdclef_effnet_b0_submission_artifacts.zip (0.00 MB)


/kaggle/working/submission.csv

/kaggle/working/birdclef_effnet_b0_submission_artifacts.zip